In [0]:
from pyspark.sql.functions import year, month, sum, countDistinct, col, count, round, avg, when
from pyspark.sql import functions as F

# Visão Comercial e Volume de Produtos

In [0]:
df_final_gold = spark.read.table('medalhao_rocket.silver.fat_pedidos_total')
df_dim_produtos_gold = spark.read.table('medalhao_rocket.silver.dim_produtos')
df_itens_pedidos_gold = spark.read.table('medalhao_rocket.silver.fat_itens_pedidos')


# Junções das tabelas dim_vendedores, fat_pedidos_total, fat_itens_pedidos, dim_produtos
# Junções realizadas através das chaves primárias id_pedido e id_produto

df_join = (
    df_final_gold
    .join(
        df_itens_pedidos_gold.alias('pd'),
        df_final_gold.id_pedido == df_itens_pedidos_gold.id_pedido,
        "inner"
    )
    .join(
        df_dim_produtos_gold.alias('prod'),
        df_itens_pedidos_gold.id_produto == df_dim_produtos_gold.id_produto,
        "inner"
    )
)

df_gold_vendas_comercial = (
    df_join
    .groupBy(
        year(col("data_pedido")).alias("ano_venda"),
        month(col("data_pedido")).alias("mes_venda"),
        col("categoria_produto")
    )
    .agg(
        countDistinct('pd.id_pedido').alias('total_pedidos'),
        count('pd.id_pedido').alias('qtd_itens_vendidos'),
        round(sum('valor_total_pago_brl'), 2).alias('receita_total_brl'),
        round(sum('valor_total_pago_usd'), 2).alias('receita_total_usd'),
        round(sum('valor_total_pago_brl') / countDistinct('pd.id_pedido'),2).alias('ticket_medio_brl')

    )
)

df_gold_vendas_comercial.write.mode("overwrite").saveAsTable("medalhao_rocket.gold.fat_vendas_comercial")
display(df_gold_vendas_comercial)

In [0]:
# Produto com mais vendas

df_dim_produtos_gold = spark.read.table('medalhao_rocket.silver.dim_produtos')
df_fat_itens_pedidos = spark.read.table('medalhao_rocket.silver.fat_itens_pedidos')

df_fat_itens_pedidos = (
    df_fat_itens_pedidos
    .join(
        df_dim_produtos_gold.alias('prod'),
        df_fat_itens_pedidos.id_produto == df_dim_produtos_gold.id_produto,
        "left"
    ).select(
        "nome_produto",
        "categoria_produto"
    )
    .groupBy(
        "nome_produto",
        "categoria_produto"
    )
    .agg(
        F.count("nome_produto").alias("qtd_vendida")
    )
    .orderBy(F.col('qtd_vendida').desc()))


df_fat_itens_pedidos = df_fat_itens_pedidos.limit(5)
display(df_fat_itens_pedidos)

In [0]:
# Produto com menos vendas

df_dim_produtos_gold = spark.read.table('medalhao_rocket.silver.dim_produtos')
df_fat_itens_pedidos = spark.read.table('medalhao_rocket.silver.fat_itens_pedidos')

df_fat_itens_pedidos = (
    df_fat_itens_pedidos
    .join(
        df_dim_produtos_gold.alias('prod'),
        df_fat_itens_pedidos.id_produto == df_dim_produtos_gold.id_produto,
        "left"

    ).select(
        "nome_produto",
        "categoria_produto"
    )
    .groupBy(
        "nome_produto",
        "categoria_produto"
    )
    .agg(
        F.count("nome_produto").alias("qtd_vendida")
    )
    .orderBy(F.col('qtd_vendida').asc()))

df_fat_itens_pedidos = df_fat_itens_pedidos.limit(5)
display(df_fat_itens_pedidos)


# Satisfação de Clientes e Qualidade de produtos

In [0]:
from pyspark.sql.functions import col, count

df_vendedor_gold = spark.read.table('medalhao_rocket.silver.dim_vendedores')
df_avaliacoes_gold = spark.read.table('medalhao_rocket.silver.fat_avaliacoes_pedidos')
df_pedidos_gold = spark.read.table('medalhao_rocket.silver.fat_itens_pedidos')
df_produtos_gold = spark.read.table('medalhao_rocket.silver.dim_produtos')

# Junções das tabelas dim_vendedores, fat_avaliacoes_pedidos, fat_itens_pedidos e dim_produtos
# Junções realizadas através das chaves primárias id_pedido, id_vendedor e id_produto

df_join = (
    df_pedidos_gold
    .join(
        df_avaliacoes_gold,
        df_avaliacoes_gold.id_pedido == df_pedidos_gold.id_pedido, 
        "inner"
    )
    .join(
        df_vendedor_gold,
        df_pedidos_gold.id_vendedor == df_vendedor_gold.id_vendedor, 
        "inner"
    )
    .join(
        df_produtos_gold,
        df_pedidos_gold.id_produto == df_produtos_gold.id_produto, 
        "left"
    )
)

# Definição dos agrupamentos e agregados
df_gold_vendas_comercial = (
    df_join
    .groupBy(
        col('categoria_produto'),
        col('nome_vendedor'),  
        col('estado')  
    )
    .agg(
        count(col('id_avaliacao')).alias('total_avaliacoes'),
        round(avg(col('nota_avaliacao')).alias('avaliacao_media'),2).alias('avaliacao_media'),
        sum(when(col('nota_avaliacao') >= 4, 1).otherwise(0)).alias('total_avaliacoes_positivas'),
        sum(when(col('nota_avaliacao') <= 2, 1).otherwise(0)).alias('total_avaliacoes_negativas'),
        (round((col('total_avaliacoes_positivas')/count(col('id_avaliacao'))),2)*100).alias('percentual_satisfacao')
    )
)

df_gold_vendas_comercial.write.mode('overwrite').saveAsTable('medalhao_rocket.gold.fat_avaliacoes_clientes')

In [0]:
# Definindo o DataFrame utilizado para as próximas visualizações

df_vendedor_gold = spark.read.table('medalhao_rocket.silver.dim_vendedores').alias('v')
df_avaliacoes_gold = spark.read.table('medalhao_rocket.silver.fat_avaliacoes_pedidos').alias('av')
df_pedidos_gold = spark.read.table('medalhao_rocket.silver.fat_itens_pedidos').alias('ped')
df_produtos_gold = spark.read.table('medalhao_rocket.silver.dim_produtos').alias('prod')

df_join = (
    df_pedidos_gold
    .join(
        df_avaliacoes_gold,
        df_avaliacoes_gold.id_pedido == df_pedidos_gold.id_pedido,
        "inner"
    )
    .join(
        df_vendedor_gold,
        df_pedidos_gold.id_vendedor == df_vendedor_gold.id_vendedor,
        "inner"
    )
    .join(
        df_produtos_gold,
        df_pedidos_gold.id_produto == df_produtos_gold.id_produto,
        "left"
    )
)

In [0]:
# Produtos com as mais altas avaliações

df_rank_produtos = (
    df_join
    .groupBy(col('nome_produto'))
    .agg(
        avg(col('nota_avaliacao')).alias('nota_media'),
        count(col('id_avaliacao')).alias('quantidade_avaliacao')
    )
    .orderBy(
        col('nota_media').desc(),
        col('quantidade_avaliacao').desc()
    )
)

df_rank_produtos = df_rank_produtos.limit(1)

display(df_rank_produtos)

In [0]:
# Produtos com as mais baixas avaliações

df_rank_produtos = (
    df_join
    .groupBy(col('nome_produto'))
    .agg(
        avg(col('nota_avaliacao')).alias('nota_media'),
        count(col('id_avaliacao')).alias('quantidade_avaliacao')
    )
    .orderBy(
        col('nota_media').asc(),
        col('quantidade_avaliacao').desc()
    )
)

df_rank_produtos = df_rank_produtos.limit(1)

display(df_rank_produtos)

In [0]:
# Produtos com as mais altas avaliações

df_rank_produtos = (
    df_join
    .groupBy(col('nome_vendedor'))
    .agg(
        avg(col('nota_avaliacao')).alias('nota_media'),
        count(col('id_avaliacao')).alias('quantidade_avaliacao')
    )
    .orderBy(
        col('nota_media').desc(),
        col('quantidade_avaliacao').desc()
    )
)

df_rank_produtos = df_rank_produtos.limit(1)

display(df_rank_produtos)

In [0]:
# Vendedores com as mais baixas avaliações

df_rank_produtos = (
    df_join
    .groupBy(col('nome_vendedor'))
    .agg(
        avg(col('nota_avaliacao')).alias('nota_media'),
        count(col('id_avaliacao')).alias('quantidade_avaliacao')
    )
    .orderBy(
        col('nota_media').asc(),
        col('quantidade_avaliacao').desc()
    )
)

df_rank_produtos = df_rank_produtos.limit(1)

display(df_rank_produtos)